# Machine Learning Workflow


In [ ]:
## How to work through this Study list
A good way to do this is:
1. Write the code for each step yourself.
2. Run one step.
3. Inspect the output.
4. Only then move to the next step.

    Recommended first version
For your fraud dataset, I would start with this:
	1.	Clean target.
	2.	Convert date/time.
	3.	Drop IDs.
	4.	Stratified split.
	5.	Weighted Random Forest.
	6.	Evaluate with confusion matrix, recall, F1, PR-AUC.
	7.	Then tune the threshold.

## Step 1: Load and inspect


In [ ]:
Step 1: Load and inspect
Use this first cell to check class imbalance before modeling. 
This tells you whether the minority class is small enough that accuracy will be misleading

**Answer:
import pandas as pd

# If already loaded, skip this line
# clean_fraud_df = pd.read_csv("your_file.csv")

df = clean_fraud_df.copy()

print(df.shape)
print(df.head())
print(df["fraud_label"].value_counts(dropna=False))
print(df["fraud_label"].value_counts(normalize=True, dropna=False))


## Step 2: Clean the target


In [ ]:
Step 2: Clean the target
"""
Convert the target into binary form so scikit-learn can train properly. 
This assumes  "Fraud"  is the positive class and  "Normal"  is the negative class
"""
**Answer:
df["fraud_label"] = df["fraud_label"].map({
    "Normal": 0,
    "Fraud": 1
})

print(df["fraud_label"].value_counts(dropna=False))


## Step 3: Basic feature engineering


In [ ]:
Step 3: Basic feature engineering

Your data has dates, times, booleans stored as  "Yes" / "No" , location columns, 
and identifiers. A simple first pass is to convert time/date into usable numeric 
features and drop columns that are likely IDs with low generalization value, 
like  customer_id ,  device_id , and  ip_address

**Answer:

df["transaction_date"] = pd.to_datetime(df["transaction_date"], errors="coerce")
df["transaction_hour"] = pd.to_datetime(df["transaction_time"], format="%H:%M", errors="coerce").dt.hour
df["transaction_day"] = df["transaction_date"].dt.day
df["transaction_month"] = df["transaction_date"].dt.month
df["transaction_dayofweek"] = df["transaction_date"].dt.dayofweek

yes_no_cols = [
    "is_international_transaction",
    "is_new_merchant",
    "unusual_time_transaction"
]

for col in yes_no_cols:
    df[col] = df[col].map({"Yes": 1, "No": 0})

drop_cols = [
    "customer_id",
    "device_id",
    "ip_address",
    "transaction_time",
    "transaction_date"
]

df = df.drop(columns=drop_cols, errors="ignore")


## Step 4: Split data


In [ ]:
## Step 4: Split data
**Goal:** separate the inputs from the thing you want to predict.  
**Hint:** create `X` for all the explanatory columns and `y` for the target column only; the target should not appear inside `X`.[3]

**Structure to follow:**
- `X = ...`
- `y = ...`

**Check yourself:** after the split, ask: “Does `X` still contain the label column?” If yes, remove it.[4]

**Answer:

df["transaction_date"] = pd.to_datetime(df["transaction_date"], errors="coerce")
df["transaction_hour"] = pd.to_datetime(df["transaction_time"], format="%H:%M", errors="coerce").dt.hour
df["transaction_day"] = df["transaction_date"].dt.day
df["transaction_month"] = df["transaction_date"].dt.month
df["transaction_dayofweek"] = df["transaction_date"].dt.dayofweek

yes_no_cols = [
    "is_international_transaction",
    "is_new_merchant",
    "unusual_time_transaction"
]

for col in yes_no_cols:
    df[col] = df[col].map({"Yes": 1, "No": 0})

drop_cols = [
    "customer_id",
    "device_id",
    "ip_address",
    "transaction_time",
    "transaction_date"
]

df = df.drop(columns=drop_cols, errors="ignore")



## Step 5: Train-test split


In [ ]:
## Step 5: Train-test split
**Goal:** keep a hidden test set for final evaluation.  

**Hint:** use a split that preserves the class ratio, 
because fraud data is often imbalanced.

For imbalanced data, use  stratify=y  
so both train and test preserve the class ratio.

**Structure to follow:**
- choose `test_size`
- set `random_state`
- include `stratify=y`

**Check yourself:** compare the class distribution in train and test. They should look similar.[5]

**Answer: 

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train target ratio:")
print(y_train.value_counts(normalize=True))
print("Test target ratio:")
print(y_test.value_counts(normalize=True))


## Step 6: Preprocessing plan


In [ ]:
## Step 6: Preprocessing plan
**Goal:** handle numeric and categorical features differently.  
**Hint:** first identify which columns are numeric, which are categorical, and which should be dropped because they are IDs or too specific to generalize well.[3]
Your dataset mixes numeric and categorical columns, so  ColumnTransformer  is the cleanest setup. Numeric columns get imputation and scaling; 
categorical columns get imputation and one-hot encoding

**Structure to follow:**
- numeric pipeline: impute missing values, then scale
- categorical pipeline: impute missing values, then one-hot encode
- combine with `ColumnTransformer`

**Check yourself:** after preprocessing, can the model accept the output without errors? If not, one of the feature types was not handled correctly.[5]

**Answer:

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_cols = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_cols),
    ("cat", categorical_pipeline, categorical_cols)
])



## Step 7: Baseline model (Train a baseline model with class weights)


In [ ]:
## Step 7: Baseline model
**Goal:** train a simple model before trying advanced balancing methods.  
"""
**Hint:** start with a model that supports class weights, because imbalance can be handled without changing the data first.[6][5]

Start with  class_weight="balanced"  before jumping to SMOTE. 
This is often the simplest and safest first fix for imbalanced classification
"""
**Structure to follow:**
- preprocessing
- model
- wrap both in a `Pipeline`

**Check yourself:** don’t look only at accuracy. 
Also inspect recall, precision, F1, and the confusion matrix.

Answer: 

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, roc_auc_score, average_precision_score
import matplotlib.pyplot as plt

baseline_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000, class_weight="balanced"))
])

baseline_model.fit(X_train, y_train)

y_pred = baseline_model.predict(X_test)
y_proba = baseline_model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))
print("PR-AUC:", average_precision_score(y_test, y_proba))

ConfusionMatrixDisplay.from_predictions(y_test, y_pred, cmap="Blues")
plt.title("Baseline Weighted Logistic Regression")
plt.show()



## Step 8: Another model for comparison (Tree model with class weights)


In [ ]:
## Step 8: Another model for comparison
**Goal:** compare against a tree-based model.  
**Hint:** use the same preprocessing, but swap the classifier. 
    Keep the comparison fair by changing only one thing at a time.[8]

Fraud data often works well with tree-based models too. 
Compare this to the weighted logistic baseline instead of assuming one is better.

**Structure to follow:**
- same train/test split
- same preprocessing
- different model
- same metrics

**Check yourself:** which model catches more fraud cases? That matters more than overall accuracy.[5]

from sklearn.ensemble import RandomForestClassifier

rf_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        class_weight="balanced",
        n_jobs=-1
    ))
])

rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)
y_proba_rf = rf_model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred_rf))
print("ROC-AUC:", roc_auc_score(y_test, y_proba_rf))
print("PR-AUC:", average_precision_score(y_test, y_proba_rf))

ConfusionMatrixDisplay.from_predictions(y_test, y_pred_rf, cmap="Greens")
plt.title("Weighted Random Forest")
plt.show()


## Step 9: Resampling (SMOTE on training data only)


In [ ]:
## Step 9: Resampling
**Goal:** help the model see more minority-class examples.  
**Hint:** do resampling only on the training set, never on the test set, 
or you’ll leak information.

If class weights are not enough, use SMOTE after preprocessing on the training data only. 
Do not apply SMOTE before splitting, and do not fit it on the test set

**Structure to follow:**
- preprocess training data
- apply SMOTE or another resampler
- train model on the resampled data

# Install once if needed:
# !pip install imbalanced-learn

**Check yourself:** if the test score looks unrealistically high, ask whether the resampler touched the test set.[5]

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

smote_model = ImbPipeline([
    ("preprocessor", preprocessor),
    ("smote", SMOTE(random_state=42)),
    ("model", LogisticRegression(max_iter=1000))
])

smote_model.fit(X_train, y_train)

y_pred_smote = smote_model.predict(X_test)
y_proba_smote = smote_model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred_smote))
print("ROC-AUC:", roc_auc_score(y_test, y_proba_smote))
print("PR-AUC:", average_precision_score(y_test, y_proba_smote))

ConfusionMatrixDisplay.from_predictions(y_test, y_pred_smote, cmap="Oranges")
plt.title("SMOTE + Logistic Regression")
plt.show()


## Step 10: Threshold tuning


In [ ]:
## Step 10: Threshold tuning
**Goal:** choose the probability cutoff that fits your problem.  
**Hint:** the default 0.5 threshold may not be the best choice for fraud detection; 
try several thresholds and compare recall vs precision.[5]

For fraud detection, recall is often more important than raw accuracy, 
so the default 0.5 cutoff may not be best. 
This cell helps you test alternative thresholds.
    
**Structure to follow:**
- get predicted probabilities
- test several thresholds
- calculate precision, recall, F1 for each

**Check yourself:** if you lower the threshold, what usually happens to recall? What happens to precision?

import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score

thresholds = np.arange(0.1, 0.91, 0.05)
results = []

for t in thresholds:
    preds = (y_proba_rf >= t).astype(int)
    results.append({
        "threshold": t,
        "precision": precision_score(y_test, preds, zero_division=0),
        "recall": recall_score(y_test, preds, zero_division=0),
        "f1": f1_score(y_test, preds, zero_division=0)
    })

threshold_df = pd.DataFrame(results)
print(threshold_df)

plt.figure(figsize=(8,5))
plt.plot(threshold_df["threshold"], threshold_df["precision"], label="Precision")
plt.plot(threshold_df["threshold"], threshold_df["recall"], label="Recall")
plt.plot(threshold_df["threshold"], threshold_df["f1"], label="F1")
plt.xlabel("Threshold")
plt.ylabel("Score")
plt.title("Threshold Tuning")
plt.legend()
plt.grid(True)
plt.show()


## Step 11: Final comparison


In [ ]:
## Step 11: Final comparison
**Goal:** decide which approach is actually best.  
**Hint:** compare weighted baseline, tree model, and resampled model using the same final test set and the same metrics.[7][5]

**Structure to follow:**
- make one results table
- include precision, recall, F1, ROC-AUC, PR-AUC
- choose based on business goal, not just one score

**Check yourself:** if your goal is fraud detection, is it better to miss fewer fraud cases or to reduce false alarms? That answer changes which metric you prioritize.[5]
